# KNN

In [4]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import KFold
from sklearn.metrics import accuracy_score, f1_score, recall_score
from sklearn.preprocessing import StandardScaler
import numpy as np
import pandas as pd

def train_knn(X, y): 

    neigborhoods = [3, 5, 7, 11, 13]
    weights = ['uniform', 'distance']

    rows = []
    for n in neigborhoods: 
        for w in weights:
            
            accs_train = []
            accs_val = []
            f1_scores = []
            recall_scores = []
            
            kf = KFold(n_splits=5)
            
            for train_index, test_index in kf.split(X): 
                xtrain, xtest = X[train_index], X[test_index]
                ytrain, ytest = y[train_index], y[test_index]
                
                scaler = StandardScaler() 
                xtrain, xtest = scaler.fit_transform(xtrain), scaler.transform(xtest)

                model = KNeighborsClassifier(
                    p=2, 
                    weights=w, 
                    n_neighbors=n
                )

                model.fit(xtrain, ytrain)

                predictions_train = model.predict(xtrain)
                predictions_val = model.predict(xtest)
                
                accs_train.append(accuracy_score(ytrain, predictions_train))
                accs_val.append(accuracy_score(ytest, predictions_val))
                f1_scores.append(f1_score(ytest, predictions_val))
                recall_scores.append(recall_score(ytest, predictions_val))
                
            rows.append({
                'numero de vecinos': n,
                'pesos': w,
                'media acc train': np.mean(accs_train), 
                'media acc validacion': np.mean(accs_val),
                'IC acc validacion': np.std(accs_val),
                'media f1': np.mean(f1_scores),
                "IC f1": np.std(f1_scores),
                'media recall': np.mean(recall_scores),
                'IC recall': np.std(recall_scores)
            })
    
    results = pd.DataFrame(rows)
    return results.sort_values(by='media f1', ascending=False)

In [6]:
# Con los datos sin filtrado 

from libs.data_configuration import get_data_partitioned

X_train, X_test, y_train, y_test = get_data_partitioned() # 20% datos para validacion por defecto 

train_knn(X_train, y_train)

,numero de vecinos,pesos,media acc train,media acc validacion,IC acc validacion,media f1,IC f1,media recall,IC recall
1,3,distance,1.000000,0.951875,0.005062,0.952405,0.004439,0.960656,0.005685
3,5,distance,1.000000,0.951125,0.004497,0.951625,0.003888,0.959474,0.008190
5,7,distance,1.000000,0.950625,0.003558,0.951057,0.002775,0.957462,0.007715
0,3,uniform,0.972438,0.948625,0.005454,0.949058,0.004741,0.954660,0.006555
7,11,distance,1.000000,0.945875,0.006195,0.946418,0.005225,0.953287,0.008102
2,5,uniform,0.960906,0.945625,0.005701,0.945919,0.005056,0.948990,0.008935
9,13,distance,1.000000,0.944250,0.004684,0.944791,0.003793,0.951974,0.007362
4,7,uniform,0.955250,0.944375,0.004524,0.944590,0.003507,0.945976,0.007054
6,11,uniform,0.946625,0.938500,0.005557,0.938777,0.004733,0.940987,0.005387
8,13,uniform,0.944531,0.935250,0.005268,0.935486,0.004118,0.936745,0.006227


In [8]:

# Con los datos filtrados 

from libs.data_configuration import get_filtered_data_partitioned

X_train, X_test, y_train, y_test = get_filtered_data_partitioned() # 20% datos para validacion por defecto 

train_knn(X_train, y_train)

,numero de vecinos,pesos,media acc train,media acc validacion,IC acc validacion,media f1,IC f1,media recall,IC recall
1,3,distance,1.000000,0.952375,0.002995,0.952582,0.002914,0.962784,0.002135
3,5,distance,1.000000,0.951250,0.003666,0.951448,0.003388,0.961056,0.005571
5,7,distance,1.000000,0.950375,0.003414,0.950586,0.003063,0.960330,0.006376
0,3,uniform,0.972781,0.949750,0.003678,0.949825,0.003625,0.957259,0.003894
2,5,uniform,0.960500,0.947625,0.004583,0.947620,0.004567,0.953521,0.008914
7,11,distance,1.000000,0.946250,0.004889,0.946321,0.004351,0.952779,0.006579
4,7,uniform,0.954812,0.944750,0.003052,0.944765,0.002677,0.950771,0.007091
9,13,distance,1.000000,0.943875,0.003566,0.943984,0.002861,0.951265,0.006174
6,11,uniform,0.947625,0.939125,0.003865,0.938907,0.003458,0.941212,0.008186
8,13,uniform,0.944625,0.936000,0.004637,0.935783,0.004154,0.938214,0.009968


Finalmente se escoge el modelo que mejor puntuacion saco, KNN con pesos basados en la distanca y 11 vecinos

In [ ]:
model = KNeighborsClassifier(
    weights='distance', 
    n_neighbors=11
)

model.fit(X_train_nrm, y_train)
predictions = model.predict(X_test_nrm)

accuracy_score(y, predictions), f1_score(y, predictions), recall_score(y, predictions)

(0.951, 0.951048951048951, 0.954864593781344)